# cloud_renderer Colab demo

Installs `cloud_renderer` straight from GitHub, downloads the shared 500-cloud dataset from Google Drive (cached locally so re-running a cell doesn't re-download), extracts one cloud into a raw `.npy` volume, renders it with the `cloud-renderer` CLI, and displays the result.

## 1. Install cloud_renderer from GitHub

In [ ]:
!pip install -q "git+https://github.com/rendervous/cloud_renderer.git"

## 2. Download and cache the cloud dataset

Downloads once to `clouds_500.h5` in the Colab runtime's local disk; subsequent runs reuse the cached file instead of re-downloading.

In [ ]:
import os

CLOUDS_FILE_ID = "1VCkqcA8t89HlMo8ZBqTNJW4eWCRyIp_n"
CLOUDS_PATH = "clouds_500.h5"

if not os.path.exists(CLOUDS_PATH):
    !pip install -q gdown
    import gdown
    gdown.download(id=CLOUDS_FILE_ID, output=CLOUDS_PATH, quiet=False)
else:
    print(f"Using cached {CLOUDS_PATH}")

## 3. Extract one cloud into a raw .npy volume

`clouds_500.h5` stores each cloud as dataset `cloud_<id>`, shape `(Z,Y,X)` - exactly what `cloud-renderer --volume_path` expects.

In [ ]:
import h5py
import numpy as np

CLOUD_ID = 0
VOLUME_PATH = f"cloud_{CLOUD_ID}.npy"

with h5py.File(CLOUDS_PATH, "r") as f:
    cloud = f[f"cloud_{CLOUD_ID}"][()].astype(np.float32)

np.save(VOLUME_PATH, cloud)
print(f"Saved {VOLUME_PATH}, shape {cloud.shape}")

## 4. Render it

Directional light only (no environment map) for this demo - see the main `README.md` for the full flag reference, including HDR environment lighting.

In [ ]:
OUTPUT_PATH = "render_output.npy"

!cloud-renderer \
    --volume_path {VOLUME_PATH} \
    --density_scale 300 \
    --light_direction 0 1 0 \
    --light_intensity 3 3 3 \
    --scattering_albedo 0.99 0.985 0.98 \
    --phase_g 0.0 \
    --bounces 32 \
    --samples 64 \
    --batch_samples 16 \
    --camera_origin 0 0.3 2.2 \
    --camera_target 0 0 0 \
    --width 512 --height 512 \
    --output_path {OUTPUT_PATH}

## 5. Show the result

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

image = np.load(OUTPUT_PATH)
srgb = np.clip(image, 0.0, 1.0) ** (1.0 / 2.2)  # render_output.npy is linear radiance - gamma-correct for display

plt.figure(figsize=(8, 8))
plt.imshow(srgb)
plt.axis("off")
plt.show()